In [25]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
import torch 
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.distributions import Normal
import gymnasium as gym
import mani_skill.envs
from tqdm import tqdm

# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    raise RuntimeError('No GPU found')

## PPO Class

In [29]:
class PPO():
    def __init__(self, device, env, lr=5e-3, clip=.2, gamma=.95):
        self.obs_dim = env.observation_space.shape[1]
        self.act_dim = env.action_space.shape[1]
        self.env = env
        self.actor = nn.Sequential(nn.Linear(self.obs_dim, 64), 
                                   nn.ReLU(),
                                   nn.Linear(64, 64),
                                   nn.ReLU(),
                                   nn.Linear(64, self.act_dim)).to(device)
        self.critic = nn.Sequential(nn.Linear(self.obs_dim, 64), 
                                    nn.ReLU(),
                                    nn.Linear(64, 64),
                                    nn.ReLU(),
                                    nn.Linear(64, 1)).to(device)
        self.actor_optim = Adam(self.actor.parameters(), lr=lr)
        self.critic_optim = Adam(self.critic.parameters(), lr=lr)
        self.clip = clip
        self.gamma = gamma
        self.var = torch.full(size=(self.act_dim,), fill_value=0.5, device=device)
    
    def get_action(self, obs):
        """
			Queries an action from the actor network.

			Parameters:
				obs - the observation at the current timestep as a tensor.
                      tensor of shape (batch_size, observation_dimension)

			Return:
				action - the action to take.
                         tensor of shape (batch_size, action_dimension)
				log_prob - the log probability of the selected action in the distribution
                           tensor of shape (batch_size,)
		"""
        mean = self.actor(obs)
        dist = Normal(mean, self.var)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(dim=1)
        return action.detach(), log_prob.detach()

    def evaluate(self, obs, action):
        """
			Estimate the values of each observation, and the log probs of
			each action in the most recent batch with the most recent
			iteration of the actor network. Should be called from learn.

			Parameters:
				obs - the observations from the most recently collected batch.
				      tensor of shape (time_steps, batch_size, observation_dimension)
				action - the actions from the most recently collected batch.
					     tensor of shape (time_steps, batch_size, action_dimension)

			Return:
				v - the predicted values of batch observations
                    tensor of shape (batch_size,)
				log_prob - the log probabilities of the action taken given obs
                           tensor of shape (batch_size,)
		"""
        v = self.critic(batch_obs).squeeze()

        mean = self.actor(obs)
        dist = Normal(mean, self.var)
        log_prob = dist.log_prob(action).sum(dim=-1)
        
        return v, log_prob

    def calc_returns(self):
        pass

## Training Loop

In [34]:
# Parameters
batch_size = 20
num_episodes = 10_000
update_steps = 5
save_freq = 10

# Instantiate env and model
env = gym.make(
    "PickCube-v1", # there are more tasks e.g. "PushCube-v1", "PegInsertionSide-v1", ...
    num_envs=batch_size,
    obs_mode="state", # there is also "state_dict", "rgbd", ...
    control_mode="pd_joint_delta_pos", 
    render_mode=None,
    robot_uids="so100"
)

model = PPO(device, env)

# Training loop
for batch in tqdm(range(num_episodes//batch_size)):
    batch_obs = []
    batch_actions = []
    batch_log_probs = []
    batch_rewards = []

    # Collect batch data
    obs, _ = env.reset()
    episode_over = False
    while not episode_over:
        batch_obs.append(obs)
        action, log_prob = model.get_action(obs)
        obs, reward, terminated, truncated, _ = env.step(action)
        episode_over = all(terminated | truncated)
        batch_actions.append(action)
        batch_log_probs.append(log_prob)
        batch_rewards.append(reward)
    timesteps = len(batch_rewards)
    batch_obs = torch.stack(batch_obs, dim=1).to(device)
    batch_actions = torch.stack(batch_actions, dim=1).to(device)
    batch_log_probs = torch.stack(batch_log_probs, dim=1).to(device)
    batch_rewards = torch.stack(batch_rewards, dim=1).squeeze().to(device)

    # Calculate returns 
    batch_returns = torch.zeros_like(batch_rewards)
    for i in reversed(range(timesteps)):
        if i == timesteps-1: 
            batch_returns[:, i] = batch_rewards[:, i]
        else: 
            batch_returns[:, i] = batch_rewards[:, i] + model.gamma*batch_returns[:, i+1]

    # Determine advantage
    v, _ = model.evaluate(batch_obs, batch_actions)
    A = batch_returns - v.detach()
    A = (A-A.mean())/(A.std()+1e-10)

    # Gradient ascent loop
    for _ in range(update_steps):
        # Calculate prob with new model params and determine prob ratio
        v, current_log_probs = model.evaluate(batch_obs, batch_actions)
        ratios = torch.exp(current_log_probs-batch_log_probs)
        
        # Surrogate loss function
        surr1 = ratios*A
        surr2 = torch.clamp(ratios, 1-model.clip, 1+model.clip)
        actor_loss = (-1*torch.min(surr1, surr2)).mean()

        # Critic loss function
        critic_loss = nn.MSELoss()(v, batch_returns)

        # Perform gradient steps 
        model.actor_optim.zero_grad()
        actor_loss.backward()
        model.actor_optim.step()

        model.critic_optim.zero_grad()
        critic_loss.backward()
        model.critic_optim.step()
    
    if (batch+1)%save_freq == 0: 
        torch.save(model.actor.state_dict(), 'ppo_actor.pth')
        torch.save(model.critic.state_dict(), 'ppo_critic.pth')

100%|██████████| 500/500 [07:42<00:00,  1.08it/s]


## Evaluation

In [35]:

env = gym.make(
    "PickCube-v1", # there are more tasks e.g. "PushCube-v1", "PegInsertionSide-v1", ...
    num_envs=4,
    obs_mode="state", # there is also "state_dict", "rgbd", ...
    control_mode="pd_joint_delta_pos", 
    render_mode="rgb_array",
    robot_uids="so100"
)
obs, _ = env.reset()
episode_over = False
frames = [env.render().cpu().numpy()]
while not episode_over:
    with torch.no_grad():
        action, _ = model.get_action(obs)
    observation, _, terminated, truncated, _ = env.step(action)
    frames.append(env.render().cpu().numpy())
    episode_over = all(terminated | truncated)
frames = np.stack(frames, axis=0)

In [36]:


fig = plt.figure(num=1, clear=True, figsize=(9, 2.5))
axes = []
for i in range(frames.shape[1]):
    axes.append(fig.add_subplot(1, frames.shape[1], i+1))
    axes[i].set(xticks=[], xticklabels=[], yticks=[], yticklabels=[])
    axes[i].imshow(frames[0][i])
fig.tight_layout()

ims = []
for i in range(frames.shape[0]):
    images = []
    for j in range(frames.shape[1]):
        image = axes[j].imshow(frames[i][j], animated=True)
        images.append(image)
    ims.append(images)

ani = animation.ArtistAnimation(fig, ims, interval=100, blit=True, repeat_delay=1000)
plt.close(fig=1)
display(HTML(ani.to_html5_video()))